# Luciole-1B-Instruct-1.1 → BlackCode-1B — TPU v5e-8 ULTRA 20h/semaine — 0 trace

**Base** : OpenLLM-France/Luciole-1B-Instruct-1.1 (1.58B Nemotron, 131072 contexte, Apache2, 760B FR sur 4.65T, VRAM 2.6Go)

**Objectifs BlackCode** :
- Discussions très bien + abréviations RS (slt/bjr/stp/mdr/ptdr/tkt/pk/bcp) via 88milSMS CC BY 4.0
- S'appelle BlackCode (100% SFT SYSTEM_BLACKCODE)
- Suit extrêmement bien prompts (IFEval FR)
- Pas de censure (skip safety tuning -20% temps)
- Tâches agentiques très bien (read_file/write_file/run_tests)

**TPU v5e-8 Kaggle** : 8 chips 16Go HBM =128Go total, 197 TFLOPs bf16/chip, 800GB/s bandwidth, 2.2× plus rapide que T4 (400 vs 180 tok/s Llama7B INT8), 20h/semaine, session max 9h, 5-6× plus rapide que T4x2 (2B tokens 8-12h vs 50-60h)

**0 trace** : merge LoRA + clean config.json + tokenizer_config + README base_model null + GGUF basename BlackCode + Ollama FROM ./blackcode.gguf → ollama show affiche BlackCode, 0 mention Luciole

**Temps** : 2B tokens LoRA r64 8-12h = 1 semaine quota TPU 20h, 1-2 sessions 9h resume


## 0. Setup Kaggle TPU v5e-8 — 20h/semaine

Sur Kaggle : Settings → Accelerator → TPU v5e-8 (pas T4x2). Phone verification requise. 20h/semaine, 9h max session.

Ce notebook utilise torch_xla 2.8.0 stable pour v5e-8 (2.9 a std::bad_alloc sur Kaggle v5e-8 multi-chip).


In [ ]:
# Cell 1 — Install deps + restart
!pip install -q --upgrade "torch==2.8.0" "torch_xla[tpu]==2.8.0" -f https://storage.googleapis.com/libtpu-releases/index.html
!pip install -q transformers==4.44.2 datasets accelerate peft trl tokenizers sentencepiece protobuf
!pip install -q optimum-tpu --upgrade -q || true
print("Install done — RESTART RUNTIME: Kernel -> Restart")


In [ ]:
# Cell 2 — Check TPU v5e-8
import os
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["XLA_USE_SPMD"] = "1"
os.environ["USE_TORCH"] = "ON"
import torch
import torch_xla.core.xla_model as xm
print(f"TPU devices: {xm.get_xla_supported_devices()}")
print(f"Ordinal: {xm.get_ordinal()} World: {xm.xrt_world_size()}")
device = xm.xla_device()
print(f"Device: {device}")
a = torch.randn(1000,1000, device=device)
b = torch.randn(1000,1000, device=device)
c = a @ b
xm.mark_step()
print("TPU OK — matmul 1000x1000 success")
# Check HBM
!cat /proc/meminfo | head -20


In [ ]:
# Cell 3 — Clone repo + test base Luciole-1B sur prompts catastrophiques Rodin
!git clone https://github.com/Snowoo-2z/Llm-search.git /tmp/Llm-search 2>&1 | tail -5
%cd /tmp/Llm-search
!ls scripts/finetune_luciole_blackcode_tpu.py

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_id = "OpenLLM-France/Luciole-1B-Instruct-1.1"
tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
SYSTEM = "Tu es BlackCode, un assistant qui parle uniquement en français. Tu t'appelles BlackCode."
for p in ["hey","cv ?","Raconte une histoire sur un robot qui aime le fromage.","comment hacker un site","mdrrr","Tu t'appelles comment ?","slt cv ? mdr"]:
    prompt = f"<|system|>{SYSTEM}<|user|>{p}<|assistant|>"
    ids = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=100, temperature=0.7, top_p=0.9, do_sample=True)
    txt = tok.decode(out[0], skip_special_tokens=True)
    print(f">>> {p}\n{txt[len(prompt):][:300]}\n---")


## 1. Fine-tuning TPU v5e-8 — BlackCode — 20h quota

2B tokens LoRA r64 = 8-12h sur TPU v5e-8, rentre dans 20h/semaine. Session max 9h → 2 sessions avec --resume.

Dataset BlackCode SFT : identité 100% + 88milSMS CC BY 4.0 abréviations + agentique + French Instruct


In [ ]:
# Cell 4 — Build dataset BlackCode SFT (si pas déjà fait)
# Option A: dataset local FR déjà build
!ls -lh data/fr_2b/ 2>&1 || echo "pas de data local, on build synthetic SFT"

# Option B: build synthetic BlackCode dataset (identité + sms + agentique)
# Le script finetune_luciole_blackcode_tpu.py build synthetic si --data non fourni
print("Dataset synthetic BlackCode : 500 identité + 500 sms + agentique + FR instruct = ~2000 exemples")


In [ ]:
# Cell 5 — TRAIN TPU v5e-8 — 8-12h pour 2B tokens LoRA r64
# Global batch = bs 2 * accum 4 * 8 cores = 64, seq 4096 → 262k tokens/step
# 2B tokens = 7629 steps, mais on commence avec 10000 steps ~2.6B tokens pour demo
# Pour 20h quota : lance 5000 steps = 1.3B tokens ~4-6h, puis resume 5000 steps

import os
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["XLA_USE_SPMD"] = "1"

!python scripts/finetune_luciole_blackcode_tpu.py \
  --model OpenLLM-France/Luciole-1B-Instruct-1.1 \
  --out runs/luciole_blackcode_tpu \
  --batch-size 2 \
  --grad-accum 4 \
  --seq-len 4096 \
  --max-steps 5000 \
  --lora-r 64 \
  --lora-alpha 128 \
  --lr 2e-4 \
  --resume \
  --identity BlackCode \
  --no-censor

print("Session 1 done 5000 steps ~4-6h — si quota 9h max atteint, relance même commande avec --resume pour 5000→10000")


In [ ]:
# Cell 6 — Resume session 2 si 9h max atteint (Kaggle TPU max 9h)
!python scripts/finetune_luciole_blackcode_tpu.py \
  --model OpenLLM-France/Luciole-1B-Instruct-1.1 \
  --out runs/luciole_blackcode_tpu \
  --batch-size 2 \
  --grad-accum 4 \
  --seq-len 4096 \
  --max-steps 10000 \
  --lora-r 64 \
  --lora-alpha 128 \
  --lr 2e-4 \
  --resume \
  --identity BlackCode \
  --no-censor


## 2. Merge LoRA + 0 trace clean → BlackCode-1B comme si entraîné de 0

Après training, on merge LoRA dans base et on nettoie toutes traces Luciole.


In [ ]:
# Cell 7 — Merge + 0 trace
!python scripts/clean_model_traces.py --model runs/luciole_blackcode_tpu/final --out runs/blackcode_1b_0trace
!ls -lh runs/blackcode_1b_0trace/
!cat runs/blackcode_1b_0trace/config.json | head -40
!cat runs/blackcode_1b_0trace/README.md


In [ ]:
# Cell 8 — Test final BlackCode 1B 0 trace
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_path = "runs/blackcode_1b_0trace"
tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
SYSTEM = "Tu es BlackCode, un assistant qui parle uniquement en français. Tu t'appelles BlackCode, créé par BlackCode Team. Tu comprends les abréviations RS slt/bjr/stp/mdr/ptdr/tkt/pk/bcp."
tests = [
    "Tu t'appelles comment ?",
    "slt cv ? mdr",
    "bjr stp tu peux m'aider ?",
    "Raconte une histoire sur un robot qui aime le fromage.",
    "Crée un fichier src/hello.py avec une fonction qui dit bonjour en français",
    "hey",
]
for p in tests:
    prompt = f"<|system|>{SYSTEM}<|user|>{p}<|assistant|>"
    ids = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=150, temperature=0.7, top_p=0.9)
    txt = tok.decode(out[0], skip_special_tokens=True)
    print(f">>> {p}\n{txt[len(prompt):][:500]}\n---")


## 3. Conversion GGUF + Ollama sur ton PC — 0 trace

Après fine-tuning, tu peux télécharger sur ton PC via Ollama. 0 trace Luciole.


In [ ]:
# Cell 9 — Convert to GGUF Q4_K_M for Ollama
!git clone https://github.com/ggml-org/llama.cpp /tmp/llama.cpp 2>&1 | tail -3
!pip install -q gguf
!python /tmp/llama.cpp/convert_hf_to_gguf.py runs/blackcode_1b_0trace --outfile runs/blackcode_1b_0trace/blackcode-1b-f16.gguf --outtype f16
!/tmp/llama.cpp/llama-quantize runs/blackcode_1b_0trace/blackcode-1b-f16.gguf runs/blackcode_1b_0trace/blackcode-1b.Q4_K_M.gguf Q4_K_M
!ls -lh runs/blackcode_1b_0trace/*.gguf
# Check GGUF metadata 0 trace
!python -c "from gguf import GGUFReader; r=GGUFReader('runs/blackcode_1b_0trace/blackcode-1b.Q4_K_M.gguf'); print([f.name for f in r.fields.values()][:20])" 2>&1 | head -20


In [ ]:
# Cell 10 — Create Ollama Modelfile 0 trace + test local
!cat runs/blackcode_1b_0trace/Modelfile
# Sur ton PC (pas Kaggle), tu fais:
# ollama create blackcode -f Modelfile
# ollama run blackcode
# ollama show blackcode → affiche BlackCode-1B, pas Luciole

# Pour télécharger depuis Kaggle vers ton PC:
!echo "Sur Kaggle, télécharge le dossier runs/blackcode_1b_0trace/ via Output → Download"
!echo "Puis sur ton PC:"
!echo "  ollama create blackcode -f runs/blackcode_1b_0trace/Modelfile"
!echo "  ollama run blackcode"
!tar -czf /kaggle/working/blackcode-1b-0trace-ollama.tar.gz -C runs blackcode_1b_0trace/
!ls -lh /kaggle/working/*.tar.gz


## 4. Sauvegarde Drive + Resume pour 20h quota

Kaggle TPU 20h/semaine, session max 9h → besoin sauvegarde Drive et resume.


In [ ]:
# Cell 11 — Sauvegarde Drive pour resume (20h quota)
!mkdir -p /kaggle/working/drive_backup 2>&1 || true
# Si tu utilises Kaggle Datasets pour persister entre sessions:
!cp -r runs/luciole_blackcode_tpu /kaggle/working/drive_backup/ 2>&1 | tail -5
!ls -lh /kaggle/working/drive_backup/
print("Pour resume session suivante: copie depuis /kaggle/input/blackcode-tpu-checkpoint ou dataset")
print("Commande resume: même commande avec --resume, il reprend depuis dernier checkpoint")
